# GPU 딥러닝 실습 — MNIST 손글씨 숫자 분류

**제출일: 2026-11-12 · 오프라인 MaintQA 정비이력 챗봇과는 별도의 실습입니다**

Google Colab에서 **T4 GPU** 런타임으로 5개 섹션을 모두 실행하고, 각 섹션의 실제 출력 화면을 증빙 01~05로 캡처합니다. 학생용 정비 DB나 개인 파일은 업로드하지 않습니다. 학습·검증·테스트 데이터는 서로 분리되어 있으며, 아래 결과는 미리 채워 넣은 값이 아니라 실제 실행으로 생성됩니다.

- 데이터셋: `torchvision.datasets.MNIST` (공개 데이터)
- 모델: 소형 합성곱 신경망(CNN)
- 고정 시드: 42
- GPU 필수 — CPU 결과는 GPU 증빙으로 제출하지 않습니다.

## 01 · GPU 연결 확인 및 재현 가능한 실행 환경

In [ ]:
import os, random, time, json, platform
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "koreanize-matplotlib"], check=True)
import matplotlib.pyplot as plt
import koreanize_matplotlib  # 그래프 한글 표시

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "GPU가 필요합니다: 런타임 > 런타임 유형 변경 > T4 GPU 선택 후 다시 실행하세요."
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device("cuda")
OUT = Path("/content/maintqa_gpu_lab_outputs"); OUT.mkdir(exist_ok=True)
run_info = {"utc_started": datetime.now(timezone.utc).isoformat(),
            "python": platform.python_version(), "torch": torch.__version__,
            "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0), "seed": SEED}
print("증빙 01 | GPU 연결 확인")
print(json.dumps(run_info, indent=2, ensure_ascii=False))
x = torch.randn(1024, 1024, device=device)
y = x @ x.T
torch.cuda.synchronize()
print("텐서 연산이 실행된 장치:", y.device)
print("GPU 메모리 사용량 (MiB):", round(torch.cuda.memory_allocated()/2**20, 1))
del x, y

## 02 · 공개 데이터 준비, 데이터 분할 검증 및 샘플 시각화

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
full_train = datasets.MNIST("/content/mnist_data", train=True, download=True, transform=transform)
test_data = datasets.MNIST("/content/mnist_data", train=False, download=True, transform=transform)
order = torch.randperm(len(full_train), generator=torch.Generator().manual_seed(SEED))
train_ids = order[:12000].tolist()
val_ids = order[12000:15000].tolist()
assert not set(train_ids).intersection(val_ids)
train_loader = DataLoader(Subset(full_train, train_ids), batch_size=128, shuffle=True,
                          generator=torch.Generator().manual_seed(SEED), num_workers=2, pin_memory=True)
val_loader = DataLoader(Subset(full_train, val_ids), batch_size=256, shuffle=False, num_workers=2)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False, num_workers=2)
print("증빙 02 | 공개 데이터 및 분할 검증")
print(f"학습: {len(train_ids):,}장 | 검증: {len(val_ids):,}장 | 별도 테스트: {len(test_data):,}장")
print("학습/검증 데이터 중복: 0건 | 입력: 흑백 1 x 28 x 28 | 클래스: 10개(숫자 0~9)")
fig, axes = plt.subplots(2, 6, figsize=(10, 3.6))
for ax, idx in zip(axes.flat, train_ids[:12]):
    ax.imshow(full_train.data[idx], cmap="gray")
    ax.set_title(f"정답: {int(full_train.targets[idx])}"); ax.axis("off")
fig.suptitle("MNIST 학습 데이터 샘플"); fig.tight_layout()
fig.savefig(OUT/"02_training_samples.png", dpi=180, bbox_inches="tight")
plt.show()

## 03 · CNN 모델 구조와 GPU 실제 학습

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(32*7*7, 64), nn.ReLU(), nn.Linear(64, 10))
    def forward(self, x): return self.layers(x)

model = SmallCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
history = []
def evaluate(loader):
    model.eval(); loss_total = correct = count = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss_total += criterion(logits, labels).item()*len(labels)
            correct += (logits.argmax(1)==labels).sum().item(); count += len(labels)
    return loss_total/count, correct/count

print("증빙 03 | GPU 실제 학습")
print(model)
print("모델이 올라간 장치:", next(model.parameters()).device)
print("학습 가능한 파라미터 수:", sum(p.numel() for p in model.parameters()))
for epoch in range(1, 4):
    started = time.perf_counter(); model.train(); total_loss = correct = count = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images); loss = criterion(logits, labels)
        loss.backward(); optimizer.step()
        total_loss += loss.item()*len(labels)
        correct += (logits.argmax(1)==labels).sum().item(); count += len(labels)
    val_loss, val_acc = evaluate(val_loader)
    torch.cuda.synchronize()
    row = dict(epoch=epoch, train_loss=total_loss/count, train_acc=correct/count,
               val_loss=val_loss, val_acc=val_acc, seconds=time.perf_counter()-started)
    history.append(row)
    print(f"에폭 {epoch}/3 | 학습 손실={row['train_loss']:.4f}, 정확도={row['train_acc']:.2%} | "
          f"검증 손실={val_loss:.4f}, 정확도={val_acc:.2%} | {row['seconds']:.1f}초", flush=True)
torch.save(model.state_dict(), OUT/"small_cnn_mnist.pt")

## 04 · 별도 테스트셋 평가 및 혼동행렬

In [ ]:
test_loss, test_acc = evaluate(test_loader)
predictions=[]; targets=[]
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        predictions.extend(model(images.to(device)).argmax(1).cpu().tolist())
        targets.extend(labels.tolist())
cm = np.zeros((10, 10), dtype=int)
for truth, predicted in zip(targets, predictions): cm[truth, predicted] += 1
assert int(cm.sum()) == len(test_data)
print("증빙 04 | 별도 테스트셋 평가")
print(f"테스트 샘플: {len(test_data):,}장 | 손실: {test_loss:.4f} | 정확도: {test_acc:.2%}")
print("테스트 데이터는 학습이나 에폭 선택에 사용하지 않았습니다.")
fig, ax = plt.subplots(figsize=(6, 4.8))
im = ax.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        ax.text(j, i, str(cm[i,j]), ha="center", va="center", fontsize=7,
                color="white" if cm[i,j]>cm.max()/2 else "black")
ax.set(xticks=range(10), yticks=range(10), xlabel="예측한 숫자", ylabel="실제 숫자",
       title=f"테스트셋 혼동행렬 | 정확도 {test_acc:.2%}")
fig.colorbar(im, ax=ax); fig.tight_layout()
fig.savefig(OUT/"04_confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()

## 05 · 학습 곡선, 예측 결과 시각화 및 결과 파일 저장

In [ ]:
print("증빙 05 | 결과 시각화 및 파일 저장")
epochs=[h["epoch"] for h in history]
fig, axes=plt.subplots(1,2,figsize=(10,3.3))
for key, label in [("train", "학습"),("val", "검증")]:
    axes[0].plot(epochs,[h[key+"_loss"] for h in history],"o-",label=label)
    axes[1].plot(epochs,[h[key+"_acc"]*100 for h in history],"o-",label=label)
axes[0].set(title="에폭별 손실",xlabel="에폭",ylabel="교차 엔트로피 손실")
axes[1].set(title="에폭별 정확도",xlabel="에폭",ylabel="정확도 (%)")
for ax in axes: ax.set_xticks(epochs); ax.legend(); ax.grid(alpha=.25)
fig.tight_layout(); fig.savefig(OUT/"05_learning_curves.png",dpi=180,bbox_inches="tight");plt.show()
fig, axes=plt.subplots(1,8,figsize=(10,2))
for idx,ax in enumerate(axes):
    ax.imshow(test_data.data[idx],cmap="gray")
    ax.set_title(f"정답 {targets[idx]} / 예측 {predictions[idx]}",fontsize=8,
                 color="green" if targets[idx]==predictions[idx] else "red");ax.axis("off")
fig.tight_layout();fig.savefig(OUT/"05_predictions.png",dpi=180,bbox_inches="tight");plt.show()
results={"environment":run_info,"train_count":len(train_ids),"validation_count":len(val_ids),
         "test_count":len(test_data),"epochs":3,"history":history,
         "test_loss":test_loss,"test_accuracy":test_acc,"confusion_matrix":cm.tolist()}
(OUT/"metrics.json").write_text(json.dumps(results,indent=2,ensure_ascii=False))
import shutil
shutil.make_archive("/content/maintqa_gpu_lab_results","zip",OUT)
print("지표, 모델 가중치, 그래프를 저장한 위치:",OUT)
print("다운로드할 파일: maintqa_gpu_lab_results.zip 및 실행된 이 노트북(.ipynb)")